# 📐 Lesson 10: Similarity & Indexing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/blob/main/notebooks/10_similarity_and_indexing.ipynb)

> **Two questions hide inside every vector search: *how* do we measure "close"? and *how* do we find the closest ones without checking everything?**

---

## 🎯 Goal
By the end of this lesson you can explain cosine similarity, dot product and Euclidean distance, and you understand why vector databases use *approximate* search and what you trade for speed.

## ℹ️ Lesson info

| | |
|---|---|
| ⏱ **Time** | ~25 minutes |
| 📚 **Prerequisites** | [Lesson 09](09_vector_databases.ipynb) |
| 🔑 **API key** | Not needed |
| 🛠 **Libraries** | `numpy`, `faiss-cpu` |

In [ ]:
!pip install -q numpy faiss-cpu

## 📏 Three ways to say "close"

| Measure | What it looks at | Bigger or smaller is closer? |
|---|---|---|
| **Cosine similarity** | the **angle** between two vectors (ignores length) | bigger |
| **Dot product** | angle **and** length together | bigger |
| **Euclidean distance (L2)** | the straight-line **distance** between the two points | smaller |

Let's see how they differ with three tiny vectors:

- `a = [1, 2, 3]`
- `b = [2, 4, 6]`: points in the **same direction** as `a`, but is twice as long
- `c = [3, 2, 1]`: points in a **different direction**, with the same length as `a`

In [1]:
import numpy as np

a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 4.0, 6.0])
c = np.array([3.0, 2.0, 1.0])

def cosine(x, y):
    return float(x @ y / (np.linalg.norm(x) * np.linalg.norm(y)))

for name, other in [("b", b), ("c", c)]:
    print(f"a vs {name}:  cosine={cosine(a, other):.3f}  dot={float(a @ other):.1f}  distance={np.linalg.norm(a - other):.3f}")

a vs b:  cosine=1.000  dot=28.0  distance=3.742
a vs c:  cosine=0.714  dot=10.0  distance=2.828


### 🤔 The three measures disagree!

- By **cosine**, `b` is a *perfect* match for `a` (same direction).
- By **dot product**, `b` scores highest too (but only because it's long).
- By **Euclidean distance**, `b` is *farther* from `a` than `c` is.

So which is "right"? It depends on whether **length** should matter. For text meaning, we usually care about the **direction**, not the length. The simplest way to remove length from the picture is to **normalize** every vector to length 1 (we did this with `normalize_embeddings=True`).

For normalized vectors, the three measures **agree on the ranking**:

`distance² = 2 − 2 × cosine`   and   `dot product = cosine`

Let's check that.

In [2]:
def normalize(x):
    return x / np.linalg.norm(x)

a_n, b_n, c_n = normalize(a), normalize(b), normalize(c)

for name, other in [("b", b_n), ("c", c_n)]:
    cos = cosine(a_n, other)
    dot = float(a_n @ other)
    dist = float(np.linalg.norm(a_n - other))
    print(f"a vs {name}:  cosine={cos:.3f}  dot={dot:.3f}  distance={dist:.3f}  2-2*cosine={2 - 2 * cos:.3f}  distance^2={dist ** 2:.3f}")

a vs b:  cosine=1.000  dot=1.000  distance=0.000  2-2*cosine=0.000  distance^2=0.000
a vs c:  cosine=0.714  dot=0.714  distance=0.756  2-2*cosine=0.571  distance^2=0.571


## 🐢 The speed problem

So far, every search compared the question with **every** vector (**brute force**, or *exact* search). That's fine for 4 chunks. But the cost grows with the number of vectors: ten times more vectors means about ten times more work per question.

The fix is an **approximate nearest neighbor (ANN)** index. It builds a smart structure ahead of time, so a search only visits a *small part* of the data. The price: it may **miss** a few of the true nearest neighbors.

One popular structure is **HNSW**: it connects vectors in a graph with several layers. A search starts at the top, takes big steps toward the right region, then refines on the bottom layer, like zooming into a map. The knob `efSearch` says *how many candidates to explore*: more is slower but finds more of the true neighbors.

To measure that trade-off we need many vectors. We build **50,000 synthetic vectors** that form groups ("topics"), the way real text embeddings tend to cluster. (Pure random vectors have no structure, and approximate search does badly on them, so they would be a misleading test.)

In [3]:
import time
import faiss

rng = np.random.default_rng(0)
N, DIM, N_QUERIES, N_TOPICS = 50_000, 384, 200, 200

topic_centers = rng.normal(size=(N_TOPICS, DIM)).astype("float32")

def make_vectors(n):
    topic = rng.integers(0, N_TOPICS, size=n)
    vectors = topic_centers[topic] + 0.6 * rng.normal(size=(n, DIM)).astype("float32")
    return (vectors / np.linalg.norm(vectors, axis=1, keepdims=True)).astype("float32")

vectors = make_vectors(N)
queries = make_vectors(N_QUERIES)
print("vectors:", vectors.shape, "| queries:", queries.shape)

flat_index = faiss.IndexFlatIP(DIM)
flat_index.add(vectors)

start = time.perf_counter()
exact_scores, exact_ids = flat_index.search(queries, 10)
flat_seconds = time.perf_counter() - start
print(f"exact search for {N_QUERIES} queries: {flat_seconds:.3f} s")

vectors: (50000, 384) | queries: (200, 384)
exact search for 200 queries: 0.208 s


## 🎯 Measuring what we lose: recall

The exact search gives us the **true** top 10 for every query. For the approximate index we ask: *of those true top 10, how many did it also return?*

`recall@10 = (true neighbors found) / 10`

`1.0` means it found all of them. `0.9` means it missed one in ten on average.

We build an HNSW index and try three `efSearch` values.

In [5]:
def best_time(function, repeats=5):
    function()  # warm-up run, not measured
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        function()
        times.append(time.perf_counter() - start)
    return min(times)

flat_seconds = best_time(lambda: flat_index.search(queries, 10))
print(f"exact search for {N_QUERIES} queries: {flat_seconds:.3f} s")

start = time.perf_counter()
hnsw_index = faiss.IndexHNSWFlat(DIM, 32, faiss.METRIC_INNER_PRODUCT)
hnsw_index.add(vectors)
print(f"HNSW build time: {time.perf_counter() - start:.1f} s")

for ef_search in [8, 16, 64]:
    hnsw_index.hnsw.efSearch = ef_search
    seconds = best_time(lambda: hnsw_index.search(queries, 10))
    approx_scores, approx_ids = hnsw_index.search(queries, 10)
    recall = np.mean([len(set(exact_ids[i]) & set(approx_ids[i])) / 10 for i in range(N_QUERIES)])
    print(f"efSearch={ef_search:3}  time={seconds:.4f} s  speedup={flat_seconds / seconds:6.1f}x  recall@10={recall:.3f}")

exact search for 200 queries: 0.143 s
HNSW build time: 1.0 s
efSearch=  8  time=0.0036 s  speedup=  39.9x  recall@10=0.879
efSearch= 16  time=0.0040 s  speedup=  35.6x  recall@10=0.951
efSearch= 64  time=0.0064 s  speedup=  22.4x  recall@10=0.993


### 👀 What we saw

On 50,000 vectors and 200 queries (on one machine):

| Search | Time | Speedup | recall@10 |
|---|---|---|---|
| Exact (brute force) | 0.143 s | 1x | 1.000 (by definition) |
| HNSW, `efSearch=8` | 0.0036 s | 40x | 0.879 |
| HNSW, `efSearch=16` | 0.0040 s | 36x | 0.951 |
| HNSW, `efSearch=64` | 0.0064 s | 22x | 0.993 |

**The trade-off in one line:** more `efSearch` means a bit slower and more accurate. Going from 8 to 64 cost about 2x the time and gained about 11 points of recall.

Two honest remarks:

- **Exact search was not slow here.** 0.143 s for 200 queries is about **0.7 ms per query**. For tens of thousands of vectors, exact search is often fast enough, and it is simpler and never misses. Approximate indexes start to matter with *much* larger collections. Measure before you decide.
- **Timing needs care.** Our first measurement said `efSearch=8` was the *slowest*, which can't be right. The cause was measuring a single short run. A **warm-up run** and the **best of several repeats** gave sensible numbers. Absolute times differ on every machine, so trust the *pattern*, not the exact values.

## 🧰 Other kinds of indexes (the idea only)

We tested only Flat (exact) and HNSW. You will meet other names, so here is the idea behind each. We did **not** run them in this lesson.

| Index | Idea | Trade-off |
|---|---|---|
| **Flat** | compare with every vector | exact, but slowest at large scale |
| **HNSW** | a multi-layer graph, walked from coarse to fine | fast and accurate, uses extra memory, slower to build |
| **IVF** | split the space into regions; search only the closest few regions | needs a "training" step; can miss neighbors near region borders |
| **PQ** (compression) | store a compact approximation of each vector | much less memory, less accurate |

Real vector databases usually pick a sensible default (Chroma uses HNSW, as we saw in its configuration in Lesson 09) and let you tune a few knobs.

## ⚠️ Things to watch out for

- **Our data is synthetic.** It's "topics plus noise", not real text embeddings. Real recall depends on your data and model, so measure on **your own vectors**.
- **Structure matters.** On purely random vectors (no groups), the same HNSW index found far fewer true neighbors in our own check. Approximate search relies on the data having structure, and real embeddings usually do.
- **Always report recall with speed.** "40x faster" means little if you don't say what accuracy you lost.
- **Index settings are part of your system.** A change in `efSearch` (or in the index) can change which chunks your RAG finds, so it can change the answers.

## 📌 Summary

- **Cosine** looks at the angle, **dot product** at angle and length, **Euclidean distance** at straight-line distance.
- For **normalized** vectors the three give the **same ranking**: `dot = cosine` and `distance² = 2 − 2 × cosine`.
- **Exact search** compares with every vector. **Approximate (ANN)** indexes like HNSW search only a small part of the data and may miss a few true neighbors.
- The trade-off is measured with **recall@k** (how many true neighbors you found) against **time**. In HNSW, `efSearch` controls it.
- For small collections, exact search can be fast enough. Measure before adding complexity.

## ✍️ Exercise (10 minutes)

1. Run the HNSW loop with `ef_search` values `[2, 4, 128, 256]`.
2. Make a small table of `efSearch`, time and recall@10.
3. At which `efSearch` does recall stop improving in a way that matters? Is the extra time worth it?

**Success:** you can say, in one sentence, which `efSearch` you would choose for this data and why.

💡 Solution: `solutions/10_similarity_and_indexing.ipynb` (try on your own first)

## 🔥 Challenge (optional)

Our topics have noise `0.6`. In `make_vectors`, change `0.6` to `2.0` (more noise, less clear groups) and rebuild everything.

1. **Predict first:** will recall at `efSearch=16` go up, down or stay the same? Why?
2. Run it. Were you right?

Then connect it to RAG: what would it mean for a real collection if its documents don't form clear groups?

## 📖 Next

We know how a vector database finds the nearest vectors, and fast. Now we put it to work: in the next lesson we build the **retrieval step** of RAG properly, with `top_k`, filters and what to do when nothing relevant exists.

**➡️ Next lesson:** [11 Basic Retrieval](11_retrieval_basics.ipynb)